Import Global setting

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb

from itertools import product

RANDOM_STATE = 42

Load train/val/test features

In [2]:
train_feat = pd.read_parquet("data/train_features.parquet")
val_feat = pd.read_parquet("data/val_features.parquet")
full_train_feat = pd.read_parquet("data/full_train_features.parquet")
test_feat = pd.read_parquet("data/full_test_features.parquet")

Feature selection

In [3]:
# These columns are not used as model inputs.
# srch_id and prop_id are kept separately for grouping/submission.
NON_FEATURE_COLS = [
    "srch_id",
    "prop_id",
    "date_time",
    "click_bool",
    "booking_bool",
    "gross_bookings_usd",
    "position",
    "relevance"
]

# Keep only numeric columns as model features.
feature_cols = [
    col for col in train_feat.columns
    if col not in NON_FEATURE_COLS
    and pd.api.types.is_numeric_dtype(train_feat[col])
]

print("Number of features:", len(feature_cols))
 
# Make sure validation has exactly the same features.
missing_in_val = set(feature_cols) - set(val_feat.columns)
extra_in_val = set(val_feat.columns) - set(train_feat.columns)

print("Number of features:", len(feature_cols))
print("Missing in validation:", missing_in_val)
print("Extra in validation:", len(extra_in_val))

print(feature_cols[:50])

Number of features: 244
Number of features: 244
Missing in validation: set()
Extra in validation: 0
['site_id', 'visitor_location_country_id', 'visitor_hist_starrating', 'visitor_hist_adr_usd', 'prop_country_id', 'prop_starrating', 'prop_review_score', 'prop_brand_bool', 'prop_location_score1', 'prop_location_score2', 'prop_log_historical_price', 'price_usd', 'promotion_flag', 'srch_destination_id', 'srch_length_of_stay', 'srch_booking_window', 'srch_adults_count', 'srch_children_count', 'srch_room_count', 'srch_saturday_night_bool', 'srch_query_affinity_score', 'orig_destination_distance', 'random_bool', 'comp1_rate', 'comp1_inv', 'comp1_rate_percent_diff', 'comp2_rate', 'comp2_inv', 'comp2_rate_percent_diff', 'comp3_rate', 'comp3_inv', 'comp3_rate_percent_diff', 'comp4_rate', 'comp4_inv', 'comp4_rate_percent_diff', 'comp5_rate', 'comp5_inv', 'comp5_rate_percent_diff', 'comp6_rate', 'comp6_inv', 'comp6_rate_percent_diff', 'comp7_rate', 'comp7_inv', 'comp7_rate_percent_diff', 'comp8_ra

Preparing Ranking Model Inputs

In [4]:
# LightGBM ranker needs rows sorted by search group to identify which rows belong to the same search.
train_feat = train_feat.sort_values("srch_id").reset_index(drop=True)
val_feat = val_feat.sort_values("srch_id").reset_index(drop=True)

X_train = train_feat[feature_cols]
y_train = train_feat["relevance"].astype(int)

X_val = val_feat[feature_cols]
y_val = val_feat["relevance"].astype(int)

# Group sizes are needed for LightGBM ranker to know how many rows belong to each search group.
group_train = train_feat.groupby("srch_id").size().to_numpy()
group_val = val_feat.groupby("srch_id").size().to_numpy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("Number of train groups:", len(group_train))
print("Number of validation groups:", len(group_val))
print("First 10 group sizes:", group_train[:10])

X_train: (3980039, 244)
X_val: (978308, 244)
Number of train groups: 159836
Number of validation groups: 39959
First 10 group sizes: [28 32 21 33 28 31 29 33 34 16]


Evaluate the ranking quality of the model using NDCG@k metric.

In [5]:
def dcg_at_k(relevances, k=5):
    """
    Computes DCG@k for one ranked list.
    """
    relevances = np.asarray(relevances)[:k]
    if len(relevances) == 0:
        return 0.0

    discounts = np.log2(np.arange(2, len(relevances) + 2))
    gains = (2 ** relevances - 1)
    return np.sum(gains / discounts)


def ndcg_at_k_for_group(y_true, y_score, k=5):
    """
    Computes NDCG@k for one search group.
    """
    order = np.argsort(-y_score)
    ranked_relevance = y_true[order]

    ideal_order = np.argsort(-y_true)
    ideal_relevance = y_true[ideal_order]

    dcg = dcg_at_k(ranked_relevance, k=k)
    idcg = dcg_at_k(ideal_relevance, k=k)

    if idcg == 0:
        return 0.0

    return dcg / idcg


def mean_ndcg_at_k(df, y_true_col, y_score_col, group_col="srch_id", k=5):
    """
    Computes mean NDCG@k over all searches.
    """

    scores = []

    for _, group in df.groupby(group_col):
        y_true = group[y_true_col].to_numpy()
        y_score = group[y_score_col].to_numpy()

        scores.append(ndcg_at_k_for_group(y_true, y_score, k=k))

    return np.mean(scores)

Binary Classification model - LGBMClassifier Model

In [6]:
# ---------------------------------------------------
# Binary classification target
# ---------------------------------------------------

y_train_cls = (
    (train_feat["click_bool"] == 1) |
    (train_feat["booking_bool"] == 1)
).astype(int)

y_val_cls = (
    (val_feat["click_bool"] == 1) |
    (val_feat["booking_bool"] == 1)
).astype(int)

# ---------------------------------------------------
# Train classifier
# ---------------------------------------------------
param_grid = {
    "num_leaves": [31, 63],
    "learning_rate": [0.05, 0.03],
    "min_child_samples": [50, 100]
}

keys = list(param_grid.keys())

experiments = [
    dict(zip(keys, values))
    for values in product(*param_grid.values())
]

classifier_results = []

for i, params in enumerate(experiments, start=1):
    print(f"Training classifier {i}/{len(experiments)}")
    print(params)

    classifier = lgb.LGBMClassifier(
        objective="binary",
        device="cpu",

        n_estimators=1000,

        subsample=0.8,
        colsample_bytree=0.8,

        random_state=RANDOM_STATE,
        n_jobs=-1,

        **params
    )

    classifier.fit(
        X_train,
        y_train_cls,
        eval_set=[(X_val, y_val_cls)],
        eval_metric="auc",

        callbacks=[
            lgb.early_stopping(50),
            lgb.log_evaluation(100)
        ]
    )

    # ---------------------------------------------------
    # Predict probabilities
    # ---------------------------------------------------

    preds = classifier.predict_proba(
            X_val,
            num_iteration=classifier.best_iteration_
        )[:, 1]


    # ---------------------------------------------------
    # Evaluate ranking quality using NDCG
    # ---------------------------------------------------

    val_classifier_eval = val_feat[
            ["srch_id", "prop_id", "relevance"]
        ].copy()

    val_classifier_eval["prediction"] = preds

    classifier_ndcg = mean_ndcg_at_k(
        val_classifier_eval,
        y_true_col="relevance",
        y_score_col="prediction",
        group_col="srch_id",
        k=5
    )

    classifier_results.append({
            "experiment": i,

            "num_leaves": params["num_leaves"],
            "learning_rate": params["learning_rate"],
            "min_child_samples": params["min_child_samples"],

            "best_iteration": classifier.best_iteration_,
            "validation_ndcg@5": classifier_ndcg
        })

classifier_results = pd.DataFrame(classifier_results)

classifier_results = classifier_results.sort_values(
        "validation_ndcg@5",
        ascending=False
    )

best_classifier_parameters = {
    "num_leaves": int(classifier_results.iloc[0]["num_leaves"]),
    "learning_rate": float(classifier_results.iloc[0]["learning_rate"]),
    "min_child_samples": int(classifier_results.iloc[0]["min_child_samples"]),
    "n_estimators": int(classifier_results.iloc[0]["best_iteration"])
}

print(best_classifier_parameters)
display(classifier_results)

Training classifier 1/8
{'num_leaves': 31, 'learning_rate': 0.05, 'min_child_samples': 50}
[LightGBM] [Info] Number of positive: 177702, number of negative: 3802337
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.538677 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 31959
[LightGBM] [Info] Number of data points in the train set: 3980039, number of used features: 238
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.044648 -> initscore=-3.063263
[LightGBM] [Info] Start training from score -3.063263
Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.751389	valid_0's binary_logloss: 0.164528
Early stopping, best iteration is:
[140]	valid_0's auc: 0.75355	valid_0's binary_logloss: 0.164307
Training classifier 2/8
{'num_leaves': 31, 'learning_rate': 0.05, 'min_child_samples': 100}
[LightGBM] [Info] Number of

,experiment,num_leaves,learning_rate,min_child_samples,best_iteration,validation_ndcg@5
6,7,63,0.03,50,193,0.380777
7,8,63,0.03,100,192,0.380670
5,6,63,0.05,100,108,0.379282
4,5,63,0.05,50,108,0.379141
3,4,31,0.03,100,249,0.377901
2,3,31,0.03,50,235,0.377414
0,1,31,0.05,50,140,0.377367
1,2,31,0.05,100,129,0.377028


LGBMRanker Model

In [7]:
# ---------------------------------------------------
# LightGBM Ranker hyperparameter tuning
# ---------------------------------------------------
# This cell keeps the same tuning structure as before, but uses the latest
# tuned feature set and the latest promising grid.
#
# Previous best results with the improved Task 3 features:
# - 511 leaves, lr=0.02, min_child=900, reg_lambda=40  -> NDCG@5 = 0.399595
# - 639 leaves, lr=0.02, min_child=1500, reg_lambda=80 -> NDCG@5 = 0.400651
# - lower learning-rate check did not beat 0.400651
#
# The best single model so far is therefore:
# num_leaves=639, learning_rate=0.02, min_child_samples=1500,
# reg_lambda=80, reg_alpha=0, subsample=0.8, colsample_bytree=0.8,
# best_iteration=388.

from itertools import product
import pandas as pd
import lightgbm as lgb

# No fixed base parameters are needed here because the grid contains all tuned parameters.
base_params = {}

# Latest small grid around the current best model.
# You can skip running this cell if you already have these results and go directly to blending.
param_grid = {
    "num_leaves": [639],
    "learning_rate": [0.01, 0.015, 0.02],
    "min_child_samples": [1500],
    "reg_lambda": [80.0, 120.0],
    "reg_alpha": [0.0],
    "subsample": [0.8],
    "colsample_bytree": [0.8]
}

# Create all parameter combinations.
keys = list(param_grid.keys())
experiments = [
    dict(zip(keys, values))
    for values in product(*param_grid.values())
]

print("Total experiments:", len(experiments))

# ---------------------------------------------------
# Run experiments
# ---------------------------------------------------

tuning_results = []

for i, params in enumerate(experiments, start=1):

    all_params = {**base_params, **params}

    print("=" * 60)
    print(f"Experiment {i}/{len(experiments)}")
    print(all_params)

    ranker = lgb.LGBMRanker(
        objective="lambdarank",
        metric="ndcg",
        ndcg_eval_at=[5],
        boosting_type="gbdt",
        device="cpu",

        # Lower learning rates may need many trees; early stopping selects the best iteration.
        n_estimators=2000,

        random_state=RANDOM_STATE,
        n_jobs=-1,

        **all_params
    )

    ranker.fit(
        X_train,
        y_train,
        group=group_train,

        eval_set=[(X_val, y_val)],
        eval_group=[group_val],
        eval_at=[5],

        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=100)
        ]
    )

    # Predict validation scores.
    preds = ranker.predict(
        X_val,
        num_iteration=ranker.best_iteration_
    )

    # Compute validation NDCG@5 using our own evaluator.
    tmp = val_feat[["srch_id", "prop_id", "relevance"]].copy()
    tmp["prediction"] = preds

    score = mean_ndcg_at_k(
        tmp,
        y_true_col="relevance",
        y_score_col="prediction",
        group_col="srch_id",
        k=5
    )

    tuning_results.append({
        "experiment": i,

        "num_leaves": all_params["num_leaves"],
        "learning_rate": all_params["learning_rate"],
        "min_child_samples": all_params["min_child_samples"],

        "reg_lambda": all_params["reg_lambda"],
        "reg_alpha": all_params["reg_alpha"],
        "subsample": all_params["subsample"],
        "colsample_bytree": all_params["colsample_bytree"],

        "best_iteration": ranker.best_iteration_,
        "validation_ndcg@5": score
    })

# ---------------------------------------------------
# Final results table
# ---------------------------------------------------

tuning_results = pd.DataFrame(tuning_results)

tuning_results = tuning_results.sort_values(
    "validation_ndcg@5",
    ascending=False
).reset_index(drop=True)

best_row = tuning_results.iloc[0]

best_parameters = {
    "num_leaves": int(best_row["num_leaves"]),
    "learning_rate": float(best_row["learning_rate"]),
    "min_child_samples": int(best_row["min_child_samples"]),

    "reg_lambda": float(best_row["reg_lambda"]),
    "reg_alpha": float(best_row["reg_alpha"]),
    "subsample": float(best_row["subsample"]),
    "colsample_bytree": float(best_row["colsample_bytree"]),

    "n_estimators": int(best_row["best_iteration"])
}

print("Best parameters from this tuning cell:")
print(best_parameters)

print("\nBest validation NDCG@5 from this tuning cell:")
print(best_row["validation_ndcg@5"])

display(tuning_results)

# Manually store the best single model found over all tuning rounds.
# This is used as model A in the blending section.
best_single_parameters = {
    "num_leaves": 639,
    "learning_rate": 0.02,
    "min_child_samples": 1500,
    "reg_lambda": 80.0,
    "reg_alpha": 0.0,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "n_estimators": 388
}

print("\nBest single model overall used for blending:")
print(best_single_parameters)


Total experiments: 6
Experiment 1/6
{'num_leaves': 639, 'learning_rate': 0.01, 'min_child_samples': 1500, 'reg_lambda': 80.0, 'reg_alpha': 0.0, 'subsample': 0.8, 'colsample_bytree': 0.8}


/opt/anaconda3/envs/dmt-a2/lib/python3.11/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.578007 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 31957
[LightGBM] [Info] Number of data points in the train set: 3980039, number of used features: 237
Training until validation scores don't improve for 50 rounds
[100]	valid_0's ndcg@5: 0.391387
[200]	valid_0's ndcg@5: 0.394678
[300]	valid_0's ndcg@5: 0.397067
[400]	valid_0's ndcg@5: 0.398799
Early stopping, best iteration is:
[411]	valid_0's ndcg@5: 0.399248


/opt/anaconda3/envs/dmt-a2/lib/python3.11/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


Experiment 2/6
{'num_leaves': 639, 'learning_rate': 0.01, 'min_child_samples': 1500, 'reg_lambda': 120.0, 'reg_alpha': 0.0, 'subsample': 0.8, 'colsample_bytree': 0.8}


...

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.615070 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 31957
[LightGBM] [Info] Number of data points in the train set: 3980039, number of used features: 237
Training until validation scores don't improve for 50 rounds
[100]	valid_0's ndcg@5: 0.393239
[200]	valid_0's ndcg@5: 0.396947
[300]	valid_0's ndcg@5: 0.398331
[400]	valid_0's ndcg@5: 0.398814
Early stopping, best iteration is:
[381]	valid_0's ndcg@5: 0.399264


/opt/anaconda3/envs/dmt-a2/lib/python3.11/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


Best parameters from this tuning cell:
{'num_leaves': 639, 'learning_rate': 0.02, 'min_child_samples': 1500, 'reg_lambda': 80.0, 'reg_alpha': 0.0, 'subsample': 0.8, 'colsample_bytree': 0.8, 'n_estimators': 388}

Best validation NDCG@5 from this tuning cell:
0.4006510355808823


,experiment,num_leaves,learning_rate,min_child_samples,reg_lambda,reg_alpha,subsample,colsample_bytree,best_iteration,validation_ndcg@5
0,5,639,0.020,1500,80.0,0.0,0.8,0.8,388,0.400651
1,2,639,0.010,1500,120.0,0.0,0.8,0.8,671,0.400584
2,4,639,0.015,1500,120.0,0.0,0.8,0.8,385,0.399451
3,3,639,0.015,1500,80.0,0.0,0.8,0.8,376,0.399409
4,6,639,0.020,1500,120.0,0.0,0.8,0.8,381,0.399264
5,1,639,0.010,1500,80.0,0.0,0.8,0.8,411,0.399248



Best single model overall used for blending:
{'num_leaves': 639, 'learning_rate': 0.02, 'min_child_samples': 1500, 'reg_lambda': 80.0, 'reg_alpha': 0.0, 'subsample': 0.8, 'colsample_bytree': 0.8, 'n_estimators': 388}


Validation blending of the strongest LightGBM Ranker models


In [8]:
# ---------------------------------------------------
# Validation blending experiment
# ---------------------------------------------------
# This cell trains several strong ranker variants on the training fold,
# predicts on the validation fold, and stores the validation predictions.
# Blending can improve robustness because different strong models may make
# slightly different ranking errors.

blend_model_params = {
    "A_best_single": {
        "num_leaves": 639,
        "learning_rate": 0.02,
        "min_child_samples": 1500,
        "reg_lambda": 80.0,
        "reg_alpha": 0.0,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "n_estimators": 388
    },
    "B_large_tree": {
        "num_leaves": 767,
        "learning_rate": 0.02,
        "min_child_samples": 1200,
        "reg_lambda": 80.0,
        "reg_alpha": 0.0,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "n_estimators": 290
    },
    "C_mid_child": {
        "num_leaves": 639,
        "learning_rate": 0.02,
        "min_child_samples": 1200,
        "reg_lambda": 80.0,
        "reg_alpha": 0.0,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "n_estimators": 300
    },
    "D_lower_lr": {
        "num_leaves": 639,
        "learning_rate": 0.01,
        "min_child_samples": 1500,
        "reg_lambda": 120.0,
        "reg_alpha": 0.0,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "n_estimators": 671
    }
}

val_blend_df = val_feat[["srch_id", "prop_id", "relevance"]].copy()
trained_blend_models = {}
single_model_scores = []

for model_name, params in blend_model_params.items():
    print("=" * 60)
    print("Training blend model:", model_name)
    print(params)

    ranker = lgb.LGBMRanker(
        objective="lambdarank",
        metric="ndcg",
        ndcg_eval_at=[5],
        boosting_type="gbdt",

        n_estimators=params["n_estimators"],
        learning_rate=params["learning_rate"],
        num_leaves=params["num_leaves"],
        max_depth=-1,
        min_child_samples=params["min_child_samples"],

        reg_lambda=params["reg_lambda"],
        reg_alpha=params["reg_alpha"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],

        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    ranker.fit(
        X_train,
        y_train,
        group=group_train
    )

    pred_col = f"pred_{model_name}"
    val_blend_df[pred_col] = ranker.predict(X_val)
    trained_blend_models[model_name] = ranker

    score = mean_ndcg_at_k(
        val_blend_df[["srch_id", "prop_id", "relevance", pred_col]],
        y_true_col="relevance",
        y_score_col=pred_col,
        group_col="srch_id",
        k=5
    )

    single_model_scores.append({
        "model": model_name,
        "validation_ndcg@5": score,
        **params
    })

single_model_scores = pd.DataFrame(single_model_scores).sort_values(
    "validation_ndcg@5",
    ascending=False
).reset_index(drop=True)

display(single_model_scores)


Training blend model: A_best_single
{'num_leaves': 639, 'learning_rate': 0.02, 'min_child_samples': 1500, 'reg_lambda': 80.0, 'reg_alpha': 0.0, 'subsample': 0.8, 'colsample_bytree': 0.8, 'n_estimators': 388}


/opt/anaconda3/envs/dmt-a2/lib/python3.11/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.629954 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 31957
[LightGBM] [Info] Number of data points in the train set: 3980039, number of used features: 237


/opt/anaconda3/envs/dmt-a2/lib/python3.11/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


Training blend model: B_large_tree
{'num_leaves': 767, 'learning_rate': 0.02, 'min_child_samples': 1200, 'reg_lambda': 80.0, 'reg_alpha': 0.0, 'subsample': 0.8, 'colsample_bytree': 0.8, 'n_estimators': 290}


...

Training blend model: D_lower_lr
{'num_leaves': 639, 'learning_rate': 0.01, 'min_child_samples': 1500, 'reg_lambda': 120.0, 'reg_alpha': 0.0, 'subsample': 0.8, 'colsample_bytree': 0.8, 'n_estimators': 671}


/opt/anaconda3/envs/dmt-a2/lib/python3.11/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.705584 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 31957
[LightGBM] [Info] Number of data points in the train set: 3980039, number of used features: 237


/opt/anaconda3/envs/dmt-a2/lib/python3.11/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


,model,validation_ndcg@5,num_leaves,learning_rate,min_child_samples,reg_lambda,reg_alpha,subsample,colsample_bytree,n_estimators
0,A_best_single,0.400651,639,0.02,1500,80.0,0.0,0.8,0.8,388
1,D_lower_lr,0.400584,639,0.01,1500,120.0,0.0,0.8,0.8,671
2,B_large_tree,0.400376,767,0.02,1200,80.0,0.0,0.8,0.8,290
3,C_mid_child,0.400348,639,0.02,1200,80.0,0.0,0.8,0.8,300


Select the best blend weights on validation


In [9]:
# ---------------------------------------------------
# Test blend weights on validation
# ---------------------------------------------------
# If the best blend is higher than the best single model, use that blend for Kaggle.
# If it is roughly equal, it may still be worth submitting because blends can be more stable.

blend_candidates = {
    "A_only": {
        "pred_A_best_single": 1.00
    },
    "ABC_equal": {
        "pred_A_best_single": 1/3,
        "pred_B_large_tree": 1/3,
        "pred_C_mid_child": 1/3
    },
    "ABC_weighted_50_25_25": {
        "pred_A_best_single": 0.50,
        "pred_B_large_tree": 0.25,
        "pred_C_mid_child": 0.25
    },
    "ABC_weighted_70_15_15": {
        "pred_A_best_single": 0.70,
        "pred_B_large_tree": 0.15,
        "pred_C_mid_child": 0.15
    },
    "ABCD_equal": {
        "pred_A_best_single": 0.25,
        "pred_B_large_tree": 0.25,
        "pred_C_mid_child": 0.25,
        "pred_D_lower_lr": 0.25
    },
    "ABCD_weighted": {
        "pred_A_best_single": 0.50,
        "pred_B_large_tree": 0.20,
        "pred_C_mid_child": 0.20,
        "pred_D_lower_lr": 0.10
    },
    "AD_weighted": {
        "pred_A_best_single": 0.70,
        "pred_D_lower_lr": 0.30
    }
}

blend_results = []

for blend_name, weights in blend_candidates.items():
    tmp = val_blend_df[["srch_id", "prop_id", "relevance"]].copy()
    tmp["blend_prediction"] = 0.0

    for pred_col, weight in weights.items():
        tmp["blend_prediction"] += weight * val_blend_df[pred_col]

    blend_score = mean_ndcg_at_k(
        tmp,
        y_true_col="relevance",
        y_score_col="blend_prediction",
        group_col="srch_id",
        k=5
    )

    blend_results.append({
        "blend_name": blend_name,
        "validation_ndcg@5": blend_score,
        "weights": weights
    })

blend_results = pd.DataFrame(blend_results).sort_values(
    "validation_ndcg@5",
    ascending=False
).reset_index(drop=True)

display(blend_results)

best_blend = blend_results.iloc[0]
best_blend_name = best_blend["blend_name"]
best_blend_weights = best_blend["weights"]

print("Best blend:")
print(best_blend_name)
print(best_blend["validation_ndcg@5"])
print(best_blend_weights)


,blend_name,validation_ndcg@5,weights
0,ABCD_equal,0.401653,"{'pred_A_best_single': 0.25, 'pred_B_large_tre..."
1,ABC_equal,0.401646,"{'pred_A_best_single': 0.3333333333333333, 'pr..."
2,ABC_weighted_50_25_25,0.401396,"{'pred_A_best_single': 0.5, 'pred_B_large_tree..."
3,ABCD_weighted,0.401145,"{'pred_A_best_single': 0.5, 'pred_B_large_tree..."
4,ABC_weighted_70_15_15,0.401067,"{'pred_A_best_single': 0.7, 'pred_B_large_tree..."
5,AD_weighted,0.400958,"{'pred_A_best_single': 0.7, 'pred_D_lower_lr':..."
6,A_only,0.400651,{'pred_A_best_single': 1.0}


Best blend:
ABCD_equal
0.401652573344826
{'pred_A_best_single': 0.25, 'pred_B_large_tree': 0.25, 'pred_C_mid_child': 0.25, 'pred_D_lower_lr': 0.25}


In [10]:
# # Rebuild feature list from full training data.
# final_feature_cols = [
#     col for col in full_train_feat.columns
#     if col not in NON_FEATURE_COLS
#     and pd.api.types.is_numeric_dtype(full_train_feat[col])
# ]

# # Ensure test has all final feature columns.
# missing_in_test = [col for col in final_feature_cols if col not in test_feat.columns]
# print("Missing features in test:", missing_in_test)

# # Sort by srch_id to ensure correct grouping for LightGBM ranker.
# full_train_feat = full_train_feat.sort_values("srch_id").reset_index(drop=True)
# test_feat = test_feat.sort_values("srch_id").reset_index(drop=True)

# X_full = full_train_feat[final_feature_cols]
# y_full = full_train_feat["relevance"].astype(int)
# group_full = full_train_feat.groupby("srch_id").size().to_numpy()

# X_test = test_feat[final_feature_cols]

# print("X_full:", X_full.shape)
# print("X_test:", X_test.shape)
# print("Number of final features:", len(final_feature_cols))

# ---------------------------------------------------
# Prepare full training and test matrices for final model
# ---------------------------------------------------
# This cell prepares the full training set and test set using the same feature columns.
# It does not train the model yet.

# Columns that should not be used as model input.
# srch_id and prop_id are identifiers.
# date_time is not used directly because we already extracted date/time features.
# click_bool, booking_bool, gross_bookings_usd, position, and relevance are target-related or unavailable in test.
NON_FEATURE_COLS = [
    "srch_id",
    "prop_id",
    "date_time",
    "click_bool",
    "booking_bool",
    "gross_bookings_usd",
    "position",
    "relevance"
]

# Rebuild feature list from full training data.
# Only numeric columns are used by the final LightGBM Ranker.
final_feature_cols = [
    col for col in full_train_feat.columns
    if col not in NON_FEATURE_COLS
    and pd.api.types.is_numeric_dtype(full_train_feat[col])
]

# Ensure test has all final feature columns.
missing_in_test = [col for col in final_feature_cols if col not in test_feat.columns]
print("Missing features in test:", missing_in_test)

if len(missing_in_test) > 0:
    raise ValueError("Some final training features are missing in the test set.")

# Sort by srch_id to ensure correct grouping for LightGBM ranker.
# LightGBM needs group sizes to match the row order.
full_train_feat = full_train_feat.sort_values("srch_id").reset_index(drop=True)
test_feat = test_feat.sort_values("srch_id").reset_index(drop=True)

# Create final training matrix, target, group sizes, and test matrix.
X_full = full_train_feat[final_feature_cols]
y_full = full_train_feat["relevance"].astype(int)
group_full = full_train_feat.groupby("srch_id").size().to_numpy()

X_test = test_feat[final_feature_cols]

print("X_full:", X_full.shape)
print("X_test:", X_test.shape)
print("Number of final features:", len(final_feature_cols))

Missing features in test: []
X_full: (4958347, 244)
X_test: (4959183, 244)
Number of final features: 244


In [11]:
# # Use the best iteration from validation if available.
# # If early stopping stopped at e.g. 350 trees, train final model with that many trees.
# best_cls_n_estimators = ranker.best_iteration_

# final_classifier = lgb.LGBMClassifier(
#     objective="binary",

#     n_estimators=best_cls_n_estimators,
#     num_leaves=best_classifier_parameters["num_leaves"],
#     learning_rate=best_classifier_parameters["learning_rate"],
#     min_child_samples=best_classifier_parameters["min_child_samples"],

#     subsample=0.8,
#     colsample_bytree=0.8,

#     random_state=RANDOM_STATE,
#     n_jobs=-1
# )

In [12]:
# ---------------------------------------------------
# Train final blended LightGBM Ranker on the full training set
# ---------------------------------------------------
# This cell trains the same models used in the validation blend, but now on
# the full training set. The test predictions are averaged using the best
# validation blend weights selected above.

# Convert validation prediction column names back to model names.
pred_col_to_model_name = {
    "pred_A_best_single": "A_best_single",
    "pred_B_large_tree": "B_large_tree",
    "pred_C_mid_child": "C_mid_child",
    "pred_D_lower_lr": "D_lower_lr"
}

# If you want to force a specific blend manually, uncomment one of these lines:
# best_blend_weights = {"pred_A_best_single": 1.00}
# best_blend_weights = {"pred_A_best_single": 0.50, "pred_B_large_tree": 0.25, "pred_C_mid_child": 0.25}

print("Training final blend using weights:")
print(best_blend_weights)

final_blend_models = {}
test_scores_blend = np.zeros(X_test.shape[0])

for pred_col, weight in best_blend_weights.items():
    model_name = pred_col_to_model_name[pred_col]
    params = blend_model_params[model_name]

    print("=" * 60)
    print("Training final blend model:", model_name)
    print("Weight:", weight)
    print(params)

    final_ranker = lgb.LGBMRanker(
        objective="lambdarank",
        metric="ndcg",
        ndcg_eval_at=[5],
        boosting_type="gbdt",

        n_estimators=params["n_estimators"],
        learning_rate=params["learning_rate"],
        num_leaves=params["num_leaves"],
        max_depth=-1,
        min_child_samples=params["min_child_samples"],

        reg_lambda=params["reg_lambda"],
        reg_alpha=params["reg_alpha"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],

        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    final_ranker.fit(
        X_full,
        y_full,
        group=group_full
    )

    model_test_scores = final_ranker.predict(X_test)
    test_scores_blend += weight * model_test_scores

    final_blend_models[model_name] = final_ranker

print("Final blended predictions created.")


Training final blend using weights:
{'pred_A_best_single': 0.25, 'pred_B_large_tree': 0.25, 'pred_C_mid_child': 0.25, 'pred_D_lower_lr': 0.25}
Training final blend model: A_best_single
Weight: 0.25
{'num_leaves': 639, 'learning_rate': 0.02, 'min_child_samples': 1500, 'reg_lambda': 80.0, 'reg_alpha': 0.0, 'subsample': 0.8, 'colsample_bytree': 0.8, 'n_estimators': 388}


/opt/anaconda3/envs/dmt-a2/lib/python3.11/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.780184 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 31974
[LightGBM] [Info] Number of data points in the train set: 4958347, number of used features: 237


/opt/anaconda3/envs/dmt-a2/lib/python3.11/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


Training final blend model: B_large_tree
Weight: 0.25
{'num_leaves': 767, 'learning_rate': 0.02, 'min_child_samples': 1200, 'reg_lambda': 80.0, 'reg_alpha': 0.0, 'subsample': 0.8, 'colsample_bytree': 0.8, 'n_estimators': 290}


...

Training final blend model: D_lower_lr
Weight: 0.25
{'num_leaves': 639, 'learning_rate': 0.01, 'min_child_samples': 1500, 'reg_lambda': 120.0, 'reg_alpha': 0.0, 'subsample': 0.8, 'colsample_bytree': 0.8, 'n_estimators': 671}


/opt/anaconda3/envs/dmt-a2/lib/python3.11/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.744004 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 31974
[LightGBM] [Info] Number of data points in the train set: 4958347, number of used features: 237


/opt/anaconda3/envs/dmt-a2/lib/python3.11/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


Final blended predictions created.


In [13]:
# Check which datetime columns are still in test_feat
test_feat.select_dtypes(include=["datetime64", "datetime64[ns]"]).columns

Index(['date_time'], dtype='str')

In [18]:
# ---------------------------------------------------
# Create Kaggle submission from blended predictions
# ---------------------------------------------------
# The file must contain one row per test search-property pair, sorted within
# each search from most likely to be booked/clicked to least likely.

submission = test_feat[["srch_id", "prop_id"]].copy()
submission["score"] = test_scores_blend

submission = submission.sort_values(
    ["srch_id", "score"],
    ascending=[True, False]
)

# Required Kaggle format: SearchId,PropertyId
# submission = submission[["srch_id", "prop_id"]].rename(columns={
#     "srch_id": "SearchId",
#     "prop_id": "PropertyId"
# })

submission = submission[["srch_id", "prop_id"]]

display(submission.head(30))

submission_path = "submission_lgbm_ranker_blend.csv"
submission.to_csv(submission_path, index=False)

print("Saved submission to:", submission_path)
print("Submission shape:", submission.shape)


,srch_id,prop_id
20,1,28181
6,1,99484
24,1,54937
27,1,61934
21,1,34263
7,1,95031
19,1,24194
23,1,50162
22,1,5543
14,1,90385


Saved submission to: submission_lgbm_ranker_blend.csv
Submission shape: (4959183, 2)


In [15]:
# ---------------------------------------------------
# Save feature importances for each final blend model
# ---------------------------------------------------
# A blend has multiple models, so we save one importance table per model and
# one combined weighted importance table.

importance_tables = []

for pred_col, weight in best_blend_weights.items():
    model_name = pred_col_to_model_name[pred_col]
    model = final_blend_models[model_name]

    model_importance = pd.DataFrame({
        "feature": model.feature_name_,
        "importance": model.feature_importances_,
        "model": model_name,
        "weight": weight
    }).sort_values("importance", ascending=False)

    importance_tables.append(model_importance)

    output_path = f"feature_importance_{model_name}.csv"
    model_importance.to_csv(output_path, index=False)
    print("Saved feature importance to", output_path)

all_feature_importance = pd.concat(importance_tables, ignore_index=True)

weighted_feature_importance = (
    all_feature_importance
    .assign(weighted_importance=lambda d: d["importance"] * d["weight"])
    .groupby("feature", as_index=False)["weighted_importance"]
    .sum()
    .sort_values("weighted_importance", ascending=False)
)

display(weighted_feature_importance.head(50))

weighted_feature_importance.to_csv("feature_importance_lgbm_ranker_blend_weighted.csv", index=False)
print("Saved weighted blend feature importance to feature_importance_lgbm_ranker_blend_weighted.csv")


Saved feature importance to feature_importance_A_best_single.csv
Saved feature importance to feature_importance_B_large_tree.csv
Saved feature importance to feature_importance_C_mid_child.csv
Saved feature importance to feature_importance_D_lower_lr.csv


,feature,weighted_importance
168,prop_id_mean_random_bool,5440.25
89,location1_diff_from_search_mean,5070.75
91,location2_diff_from_prop_mean,3989.75
100,log_price_diff_from_prop_mean,3953.25
120,price_pct_rank_in_search,3922.25
176,prop_id_std_location_score2,3818.50
171,prop_id_mean_total_guests,3727.25
229,star_diff_from_search_mean,3648.25
92,location2_diff_from_search_mean,3636.25
231,star_pct_rank_in_search,3537.25


Saved weighted blend feature importance to feature_importance_lgbm_ranker_blend_weighted.csv
